In [0]:
from pyspark.sql.types import *
schema= StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

source_path = "/Volumes/spakr_data/default/datastorage/AutoloaderData/bronze/"
target_path = "/Volumes/spakr_data/default/datastorage/AutoloaderData/Silver/"
checkpoint_path = "/Volumes/spakr_data/default/datastorage/AutoloaderData/checkpointLocation/"

In [0]:
df=(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(schema)
    .load(source_path))


In [0]:
display(
  df,
  checkpointLocation=checkpoint_path
)

In [0]:
for stream in spark.streams.active:
    stream.stop()

In [0]:
df.writeStream.format("delta")\
.option("checkpointLocation", checkpoint_path)\
.outputMode("append")\
.trigger(availableNow=True)\
.start(target_path)

In [0]:
query = (
  df.writeStream
    .format("delta")
    .queryName("my_stream")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start(target_path)
)

display(spark.sql("SELECT * FROM my_stream"))